In [7]:
from azure.cognitiveservices.speech import SpeechConfig, SpeechSynthesizer

# Configuration avec votre clé Azure (créez-la sur portal.azure.com)
speech_config = SpeechConfig(
    subscription="votre-clé-api-ici",  # Ex: "a1b2c3d4e5f6g7h8i9j0"
    region="eastus"                   # "eastus" ou "westeurope"
)

# Voix marocaine native (choisissez l'une ou l'autre)
speech_config.speech_synthesis_voice_name = "ar-MA-JamalNeural"  # Masculin
# speech_config.speech_synthesis_voice_name = "ar-MA-MounaNeural"  # Féminin

# Synthèse du texte
synthesizer = SpeechSynthesizer(speech_config)
result = synthesizer.speak_text_async("بغيت ناكول كسكس ف مرّاكش").get()

# Sauvegarde du fichier audio
with open("darija.wav", "wb") as audio_file:
    audio_file.write(result.audio_data)

print("✅ Fichier audio généré : darija.wav")

ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1342:(snd_func_refer) error evaluating name
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5727:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM default


✅ Fichier audio généré : darija.wav


In [5]:
# Cellule 2 : Chargement du modèle et fonction de traduction

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import os

# --- CONFIGURATION ---
# Chemin vers votre modèle local, fusionné et autonome
MODEL_PATH = "../nllb-darija-merged-model/"
# ---------------------

# Vérifier si le chemin du modèle existe
if not os.path.isdir(MODEL_PATH):
    raise FileNotFoundError(
        f"ERREUR : Le dossier du modèle '{MODEL_PATH}' est introuvable. "
        "Assurez-vous que le notebook est bien à la racine du projet et que le modèle a été fusionné."
    )

print(f"🔄 Chargement du tokenizer et du modèle depuis : {MODEL_PATH}")

# 1. Charger directement le tokenizer et le modèle final
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)

# 2. Déterminer le device (GPU si disponible, sinon CPU)
device = 0 if torch.cuda.is_available() else -1
device_name = "cuda:0" if device == 0 else "cpu"
print(f"✅ Modèle et tokenizer chargés ! Utilisation du device : {device_name}")

# 3. Créer la pipeline de traduction de Hugging Face
# C'est une manière propre et optimisée de gérer l'inférence.
translator_pipeline = pipeline(
    "translation",
    model=model,
    tokenizer=tokenizer,
    device=device
)

print("\n🚀 Le traducteur est prêt ! Vous pouvez maintenant utiliser la cellule suivante pour traduire du texte.")

🔄 Chargement du tokenizer et du modèle depuis : ../nllb-darija-merged-model/


/projets/darija_app_final/venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Modèle et tokenizer chargés ! Utilisation du device : cuda:0

🚀 Le traducteur est prêt ! Vous pouvez maintenant utiliser la cellule suivante pour traduire du texte.


In [7]:
# Cellule 3 : Test de traduction interactif

# --- MODIFIEZ CES VARIABLES POUR TESTER ---
text_to_translate = "How much does that cost"
source_lang = "eng_Latn"  # Options: "fra_Latn", "eng_Latn", "ary_Arab"
target_lang = "ary_Arab"  # Options: "fra_Latn", "eng_Latn", "ary_Arab"
# ------------------------------------------

print(f"📝 Traduction de '{text_to_translate}' ({source_lang} -> {target_lang})...")

# Appel de la pipeline
outputs = translator_pipeline(
    text_to_translate,
    src_lang=source_lang,
    tgt_lang=target_lang,
    max_new_tokens=100  # Limite la longueur de la réponse
)

# Extraire et afficher le résultat
translated_text = outputs[0]['translation_text']

print("\n🎉 Résultat :")
print(f"   {translated_text}")

📝 Traduction de 'How much does that cost' (eng_Latn -> ary_Arab)...

🎉 Résultat :
   شحال كايكلف هادشي


In [3]:
# Cellule pour tester l'API d'inférence Hugging Face

import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

print("--- Test de l'API d'inférence Hugging Face ---")

# 1. Charger les variables d'environnement (notamment HF_TOKEN)
#    load_dotenv() cherchera un fichier .env dans le dossier courant ou les dossiers parents.
if load_dotenv():
    print("✅ Fichier .env chargé avec succès.")
else:
    print("⚠️ Fichier .env non trouvé. Assurez-vous qu'il existe et contient votre HF_TOKEN.")

# 2. Récupérer le token depuis les variables d'environnement
hf_token = os.getenv("HUGGINGFACE_TOKEN")
if not hf_token:
    raise ValueError("Le token Hugging Face (HF_TOKEN) n'a pas été trouvé dans l'environnement.")
else:
    print("🔑 Token Hugging Face trouvé.")

# 3. Définir le modèle à utiliser (le même que dans votre API)
model_id = "Farid59/nllb-darija-fr_eng"
print(f"🎯 Modèle cible : {model_id}")

# 4. Initialiser le client d'inférence
#    C'est cet objet qui va communiquer avec les serveurs de Hugging Face.
try:
    client = InferenceClient(model=model_id, token=hf_token)
    print("🚀 Client d'inférence initialisé avec succès.")
except Exception as e:
    print(f"❌ Erreur lors de l'initialisation du client : {e}")
    # Stopper ici si l'initialisation échoue
    raise

# 5. Préparer les données pour le test de traduction
texte_a_traduire = "Bonjour, comment ça va aujourd'hui ?"
langue_source = "fra_Latn"  # Français
langue_cible = "ary_Arab"   # Darija

print(f"\n--- Lancement de la traduction ---")
print(f"Texte d'entrée ({langue_source}): '{texte_a_traduire}'")

# 6. Appeler l'API pour effectuer la traduction
try:
    # C'est ici que l'appel réseau vers Hugging Face a lieu.
    response = client.translation(
        texte_a_traduire,
        src_lang=langue_source,
        tgt_lang=langue_cible
    )

    # 7. Afficher le résultat
    print("\n--- Résultat de la traduction ---")
    
    # La réponse est un dictionnaire, comme celui-ci :
    # {'translation_text': 'السلام، كيف حالك اليوم؟'}
    print(f"Type de réponse reçu : {type(response)}")
    print(f"Contenu de la réponse : {response}")
    
    traduction = response.get("translation_text")
    if traduction:
        print(f"\n✅ Traduction ({langue_cible}): '{traduction}'")
    else:
        print("❌ La traduction n'a pas pu être extraite de la réponse.")

except Exception as e:
    print(f"\n❌ Une erreur est survenue lors de l'appel à l'API de traduction : {e}")
    print("   Vérifiez les points suivants :")
    print("   - Votre token HF_TOKEN est-il valide et a-t-il les permissions nécessaires ?")
    print("   - Le modèle 'Farid59/nllb-darija-fr_eng' est-il bien public sur le Hub ?")
    print("   - Avez-vous une connexion Internet active ?")

--- Test de l'API d'inférence Hugging Face ---
✅ Fichier .env chargé avec succès.
🔑 Token Hugging Face trouvé.
🎯 Modèle cible : Farid59/nllb-darija-fr_eng
🚀 Client d'inférence initialisé avec succès.

--- Lancement de la traduction ---
Texte d'entrée (fra_Latn): 'Bonjour, comment ça va aujourd'hui ?'

❌ Une erreur est survenue lors de l'appel à l'API de traduction : 
   Vérifiez les points suivants :
   - Votre token HF_TOKEN est-il valide et a-t-il les permissions nécessaires ?
   - Le modèle 'Farid59/nllb-darija-fr_eng' est-il bien public sur le Hub ?
   - Avez-vous une connexion Internet active ?


In [2]:
import requests

def query(payload):
	headers = {
		"Accept" : "application/json",
		"Authorization": "Bearer hf_xxxxxx",
		"Content-Type": "application/json" 
	}
	response = requests.post(
		"https://hj7k42o5dgbmnhl1.eu-west-1.aws.endpoints.huggingface.cloud", 
		headers=headers, 
		json=payload
	)
	return response.json()

output = query({
	"inputs": "Bonjour",
	"parameters": {
		"src_lang": "fra_Latn",
		"tgt_lang": "ary_Arab"
	}
})

print(output)

{'error': '401 Unauthorized'}
